In [14]:
import pandas as pd
import numpy as np

df_events_records = pd.read_json(
    "data/events_records.json"
)
df_events_lines = pd.read_json(
    "data/events_lines.json",
    lines=True
)

if "event_id" not in df_events_records.columns:
    df_events_records = df_events_records.T
    df_events_records.columns = [
        "event_id", "user_id", "event_time",
        "event_type", "product_id"
    ]

df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

df_events["event_time"] = pd.to_datetime(
    df_events["event_time"],
    errors="coerce"
)

df_events = df_events.dropna(
    subset=["event_time", "event_type"]
)

df_events["weekday"] = df_events["event_time"].dt.day_name()

print(df_events.head())

  event_id user_id          event_time   event_type   product_id page  \
0        1    1085 2024-01-30 20:37:32    page_view  /product/26  NaN   
1        2    1006 2024-02-24 22:55:13  add_to_cart  /product/28  NaN   
2        3    1024 2024-02-14 17:49:43        click  /product/27  NaN   
3        4    1062 2024-02-16 11:21:32        click   /product/1  NaN   
4        5    1012 2024-02-08 05:25:21  add_to_cart            /  NaN   

     weekday  
0    Tuesday  
1   Saturday  
2  Wednesday  
3     Friday  
4   Thursday  


In [11]:
#Задание 1
pivot_table = pd.pivot_table(
    df_events,
    index="weekday",
    columns="event_type",
    values="event_id",
    aggfunc="count",
    fill_value=0
)

days_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

pivot_table = pivot_table.reindex(days_order, fill_value=0)

print(pivot_table)


event_type  add_to_cart  click  page_view  purchase  scroll
weekday                                                    
Monday              114    135        118       120     117
Tuesday              99    121        121       110     102
Wednesday           121    115        101       118      89
Thursday            118    123        127       121     129
Friday              114    114        117       105     110
Saturday            101     98        107       119     122
Sunday              115    113        113       114     119


In [12]:
#Задание 2
daily_shares = pivot_table.div(
    pivot_table.sum(axis=1),
    axis=0
)

print((daily_shares * 100).round(2))

event_type  add_to_cart  click  page_view  purchase  scroll
weekday                                                    
Monday            18.87  22.35      19.54     19.87   19.37
Tuesday           17.90  21.88      21.88     19.89   18.44
Wednesday         22.24  21.14      18.57     21.69   16.36
Thursday          19.09  19.90      20.55     19.58   20.87
Friday            20.36  20.36      20.89     18.75   19.64
Saturday          18.46  17.92      19.56     21.76   22.30
Sunday            20.03  19.69      19.69     19.86   20.73


In [13]:
#Задание 3
max_day = daily_shares["add_to_cart"].idxmax()
max_share = daily_shares["add_to_cart"].max()

print("День недели:", max_day)
print("Максимальная доля add_to_cart:",
      round(max_share * 100, 2), "%")

День недели: Wednesday
Максимальная доля add_to_cart: 22.24 %
